## A/B тестирование

### Сценарий эксперимента

#### Проблема  
Продукт продаёт подписки через пейволл (экран оплаты). Текущая конверсия из просмотра пейволла в покупку составляет около 10%. Команда гипотетически теряет пользователей, которые готовы платить, но считают, что базовая цена слишком высокая, или не видят ценности в коротких планах.  
#### Гипотеза  
Если добавить на экран подписки специальное предложение (скидка для годовой подписки), то конверсия в покупку вырастет с 10% до 12.5% (относительный рост +25%), при этом общий ARPU (средняя выручка на пользователя) не упадёт из-за скидки.  
#### Группы эксперимента  
Group A (Control): Стандартный экран оплаты.  
Group B (Treatment): Экран оплаты со скидкой / акцентом на годовой план.  
#### Дерево метрик теста  
**Primary metric:**  
Paywall Conversion Rate = $\frac{количество\ пользователей\ с\ событием\ subscription\_purchase}{количество\ пользователей\ с\ событием\ paywall\_view}$  
**Secondary metric:**  
ARPU = $\frac{Сумма\ completed\_orders}{Общее\ число\ пользователей\ в\ группе}$   
**Guardrail Metrics**  
D7 Retention Rate (чтобы убедиться, что новый экран не привлекает "нецелевую" аудиторию, которая удаляет приложение на следующий день)  
Failed Payment Rate = $\frac{Число\ транзакций\ со\ статусом\ failed}{Все\ транзакции}$ (проверка корректности работы оплаты)

### Дизайн эксперимента

$H_0$: разницы между группами A и B нет  
$H_1$: разница есть  
$\alpha$ (вероятность false positive) = 0.05, $\beta$ (вероятность false negative) = 0.2  
**MDE**: базовая конверсия группы A $p_1$ = 0.1, ожидаемая конверсия группы B $p_2$ = 0.125; абсолютный MDE = 2.5%, относительный MDE = 12.5% 

расчёт размера выборки, требуемой для каждой группы
$$n = \frac{\left(Z_{\alpha/2} \cdot \sqrt{2 \cdot \bar{p}(1 - \bar{p})} + Z_{\beta} \cdot \sqrt{p_1(1 - p_1) + p_2(1 - p_2)}\right)^2}{(p_2 - p_1)^2}$$

$p_1 = 0.10$ (базовая конверсия).  
$p_2 = 0.125$ (ожидаемая конверсия).  
$\bar{p} = \frac{p_1 + p_2}{2} = 0.1125$.  
$Z_{\alpha/2} \approx 1.96$ (для $\alpha = 0.05$).  
$Z_{\beta} \approx 0.84$ (для мощности $80\%$).

In [175]:
import random
import os
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [176]:
(1.96 * np.sqrt(2 * 0.1125 * (1 - 0.1125)) + 0.84 * np.sqrt(0.1 * 0.9 + 0.125 * 0.875)) ** 2 / (0.025) ** 2

2503.7036776820173

всего в датасете 12 000 пользователей - достаточно

### Статистический анализ

#### Выгружаем данные

In [177]:
load_dotenv()

DB_USER = "postgres"
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "product_analytics"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

np.random.seed(42)
random.seed(42)

engine = create_engine(DATABASE_URL)

In [178]:
query = """
WITH user_events AS (
    -- Агрегируем события строго до 1 строки на user_id
    SELECT 
        user_id,
        MAX(CASE WHEN event_name = 'paywall_view' THEN 1 ELSE 0 END) AS viewed_paywall,
        MAX(CASE WHEN event_name = 'subscription_purchase' THEN 1 ELSE 0 END) AS made_purchase
    FROM events
    GROUP BY user_id
),
user_revenue AS (
    -- Считаем чистую выручку строго до 1 строки на user_id
    SELECT 
        user_id,
        SUM(amount) AS revenue
    FROM orders
    WHERE status = 'completed'
    GROUP BY user_id
)
SELECT 
    u.user_id,
    ab.group_id,
    COALESCE(ue.viewed_paywall, 0) AS viewed_paywall,
    COALESCE(ue.made_purchase, 0) AS made_purchase,
    COALESCE(ur.revenue, 0) AS revenue
FROM users u
JOIN ab_experiments ab ON u.user_id = ab.user_id
LEFT JOIN user_events ue ON u.user_id = ue.user_id
LEFT JOIN user_revenue ur ON u.user_id = ur.user_id
WHERE ab.experiment_name = 'paywall_discount_v1';
"""

df = pd.read_sql(query, engine)
df.head()

,user_id,group_id,viewed_paywall,made_purchase,revenue
0,6b2adfa8-3388-447d-9b9c-84f4b5987f2d,B,0,0,0.00
1,3fc98ba2-6dcf-4008-8eac-eca859f2e4ac,B,0,0,0.00
2,dbd47f02-11dc-4637-9e71-530c990e2422,B,1,1,34.99
3,38318d53-f577-4f3c-afc4-492c8867d2dc,B,1,0,0.00
4,17190b9c-7423-41df-aa45-eb2f0b6ef6f5,B,1,0,0.00


In [179]:
df = df[df['viewed_paywall'] == 1]

#### Проверка Sample Ratio Mismatch

используем критерий Хи-квадрат о соответствии

In [180]:
observed = df['group_id'].value_counts()
expected = [len(df) / 2, len(df) / 2]

chi2_srm, p_val_srm = stats.chisquare(f_obs=observed, f_exp=expected)

print(observed, '\n pvalue =', round(p_val_srm.item(), 3))
if p_val_srm < 0.01:
    print("SRM detected")
else:
    print("No SRM detected")

group_id
B    4107
A    4034
Name: count, dtype: int64 
 pvalue = 0.418
No SRM detected


#### Анализ конверсии

In [181]:
conversions = df.groupby('group_id')['made_purchase'].agg(['sum', 'count', 'mean'])
conversions['conversion_rate_%'] = conversions['mean'] * 100
print(conversions)

          sum  count      mean  conversion_rate_%
group_id                                         
A         415   4034  0.102876          10.287556
B         517   4107  0.125883          12.588264


In [ ]:
count_a = conversions.loc['A', 'sum']
count_b = conversions.loc['B', 'sum']
nobs_a = conversions.loc['A', 'count']
nobs_b = conversions.loc['B', 'count']

p_a = conversions.loc['A', 'mean']
p_b = conversions.loc['B', 'mean']

# Z-статистика и p-value
z_stat, p_val_z = proportions_ztest([count_b, count_a], [nobs_b, nobs_a])

# расчёт доверительного интервала
p_diff = p_b - p_a # центральная точка интервала
se_diff = np.sqrt((p_a * (1 - p_a) / nobs_a) + (p_b * (1 - p_b) / nobs_b)) # стандартное отклонение
z_critical = stats.norm.ppf(0.975)

ci_diff_lower = p_diff - z_critical * se_diff
ci_diff_upper = p_diff + z_critical * se_diff

print("--- Z-test & Confidence Interval ---")
print(f"CR Group A: {p_a*100:.2f}% | CR Group B: {p_b*100:.2f}%")
print(f"Absolute Lift (p_B - p_A): {p_diff*100:+.2f}%")
print(f"95% CI for Difference: [{ci_diff_lower*100:+.2f}%, {ci_diff_upper*100:+.2f}%]")
print(f"Z-statistic: {z_stat:.4f}, p-value: {p_val_z:.4e}\n")

=== Z-test & Confidence Interval ===
CR Group A: 10.29% | CR Group B: 12.59%
Absolute Lift (p_B - p_A): +2.30%
95% CI for Difference: [+0.92%, +3.68%]
Z-statistic: 3.2598, p-value: 1.1151e-03



Для таблицы сопряжённости 2 * 2 двухсторонний двухвыборочный Z-тест и критерий Хи-квадрат математически идентичны. Посмотрим, так ли это на нашем датасете:

In [ ]:
# критерий хи-квадрат
contingency_table = [
    [count_a, nobs_a - count_a],
    [count_b, nobs_b - count_b]
]

chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table, correction=False)

print("--- Chi-Square Test ---")
print(f"Contingency Table:\n{np.array(contingency_table)}")
print(f"Chi2-statistic: {chi2_stat:.4f}, p-value: {p_val_chi2:.4e}")

=== 2. Chi-Square Test ===
Contingency Table:
[[ 415 3619]
 [ 517 3590]]
Chi2-statistic: 10.6260, p-value: 1.1151e-03


#### Анализ ARPU через Bootstrap

У метрики сильная скошенность - большая часть пользователей ничего не платит, оставшаяся часть платит фиксированные суммы. Кроме этого, у нас большое число выбросов - некоторые пользователи могут платить много и из-за этого сдвигать среднее значение. Т-критерий Стьюдента может дать искаженную оценку, поэтому используем Bootstrap.

In [184]:
def bootstrap_arpu_diff(group_a, group_b, n_iterations=5000, ci=95):
    diffs = []
    n_a, n_b = len(group_a), len(group_b)
    
    np.random.seed(67)
    for _ in range(n_iterations):
        sample_a = np.random.choice(group_a, size=n_a, replace=True)
        sample_b = np.random.choice(group_b, size=n_b, replace=True)
        diffs.append(np.mean(sample_b) - np.mean(sample_a))
        
    lower_p = (100 - ci) / 2
    upper_p = 100 - lower_p
    ci_lower = np.percentile(diffs, lower_p)
    ci_upper = np.percentile(diffs, upper_p)
    
    return np.mean(diffs), (ci_lower, ci_upper)

rev_a = df[df['group_id'] == 'A']['revenue'].values
rev_b = df[df['group_id'] == 'B']['revenue'].values

mean_diff, (ci_low, ci_high) = bootstrap_arpu_diff(rev_a, rev_b)

print("\n--- ARPU Bootstrap Analysis ---")
print(f"Mean ARPU Group A: ${np.mean(rev_a):.2f}")
print(f"Mean ARPU Group B: ${np.mean(rev_b):.2f}")
print(f"ARPU Lift (B - A): ${mean_diff:.2f}")
print(f"95% Confidence Interval for ARPU Lift: [${ci_low:.2f}, ${ci_high:.2f}]")


--- ARPU Bootstrap Analysis ---
Mean ARPU Group A: $1.99
Mean ARPU Group B: $2.70
ARPU Lift (B - A): $0.71
95% Confidence Interval for ARPU Lift: [$0.36, $1.06]


### Проверка Guardrail-метрик

#### D7 retention rate

In [185]:
query = '''
select
    u.user_id,
    ab.group_id,
    max(case 
        when e.event_name = 'session_start' 
             and e.event_timestamp >= u.install_date + INTERVAL '7 days'
             and e.event_timestamp < u.install_date + INTERVAL '8 days'
        then 1 else 0 
    end) as retained_d7
from users u
inner join ab_experiments ab on u.user_id = ab.user_id
left join events e on u.user_id = e.user_id
left join orders o on u.user_id = o.user_id
where ab.experiment_name = 'paywall_discount_v1'
group by u.user_id, ab.group_id, u.install_date;
'''

df = pd.read_sql(query, engine)

In [186]:
d7_summary = df.groupby('group_id')['retained_d7'].agg(['sum', 'count', 'mean'])
d7_summary['retention_rate_%'] = d7_summary['mean'] * 100
print(d7_summary)

          sum  count      mean  retention_rate_%
group_id                                        
A         659   5961  0.110552         11.055192
B         622   6039  0.102997         10.299718


In [187]:
ret_a_success = d7_summary.loc['A', 'sum']
ret_a_total = d7_summary.loc['A', 'count']
ret_b_success = d7_summary.loc['B', 'sum']
ret_b_total = d7_summary.loc['B', 'count']

z_stat_d7, p_val_d7 = proportions_ztest([ret_b_success, ret_a_success], [ret_b_total, ret_a_total])

print(f"\nZ-stat (D7 Retention): {z_stat_d7:.4f} | p-value: {p_val_d7:.4f}")

if p_val_d7 < 0.05:
    print("обнаружены статистические изменения в удержании")
else:
    print("новый экран не ухудшил D7")


Z-stat (D7 Retention): -1.3400 | p-value: 0.1802
новый экран не ухудшил D7


#### Failed payment rate

In [188]:
query = '''
select group_id, status, count(*) as cnt
from ab_experiments
inner join orders
    using(user_id)
group by group_id, status
'''

df = pd.read_sql(query, engine)

In [189]:
a_count = df[(df['group_id'] == 'A') & (df['status'] == 'completed')]['cnt'].item()
a_failed = df[(df['group_id'] == 'A') & (df['status'] == 'failed')]['cnt'].item()
b_count = df[(df['group_id'] == 'B') & (df['status'] == 'completed')]['cnt'].item()
b_failed = df[(df['group_id'] == 'B') & (df['status'] == 'failed')]['cnt'].item()

z_stat_fpr, p_val_fpr = proportions_ztest([b_failed, a_failed], [b_count + b_failed, a_count + a_failed])

if p_val_fpr < 0.05:
    print('случился сбой в оплате')
else:
    print('сбоя в оплате не случилось')

сбоя в оплате не случилось


### Выводы  
- SRM: не обнаружен, сплитирование корректно  
- Primary metric: конверсия в группе B увеличилась с 10.29% до 12.59% (абсолютный прирост +2.30%). Разница статистически значима (Z-statistic: 3.2598, p-value: 1.1151e-03)  
- Secondary metric: средняя выручка на пользователя, дошедщего до paywall, выросла с $1.99 до $2.70. 95%-ный интервал бутстрепа [$0.36, $1.06], что подтвержает экономическую выгоду.

Рекомендация: успешный эксперимент. Выкатить обновленный экран подписки (Group B) на 100% пользователей.